# Run OQMD Downloader (Colab)
Keep notebook simple; logic lives in src/.


In [ ]:
import os
import shutil
import sys

REPO_URL = "https://github.com/malaravanthulasi19-spec/oqmd-half-heusler-colab.git"
REPO_DIR = "/content/oqmd-half-heusler-colab"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -r requirements.txt


In [ ]:
from src.config import DEFAULT_DB_PATH
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

backup_db = '/content/drive/MyDrive/oqmd_backups/oqmd_half_heusler.db'
if os.path.exists(backup_db):
    os.makedirs('/content/data', exist_ok=True)
    shutil.copy2(backup_db, DEFAULT_DB_PATH)
    print(f'Restored DB from {backup_db} to {DEFAULT_DB_PATH}')
else:
    print(f'No backup found at: {backup_db}')


In [ ]:
from src.config import DEFAULT_DB_PATH, DEFAULT_CSV_PATH, DEFAULT_PARQUET_PATH
from src.database import OQMDDatabase
from src.downloader import OQMDDownloader
from src.export_data import export_all
from src.oqmd_client import OQMDClient

db = OQMDDatabase(DEFAULT_DB_PATH)
client = OQMDClient(timeout_seconds=20)
downloader = OQMDDownloader(db=db, client=client)
result = downloader.run()
print(result)


In [ ]:
import sqlite3

with sqlite3.connect(DEFAULT_DB_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(DISTINCT entry_id) FROM oqmd_rows').fetchone()[0]
    state = conn.execute('SELECT last_offset, current_filter FROM download_state WHERE id = 1').fetchone()

last_offset = state[0] if state else 0
current_filter = state[1] if state else '(none)'

print(f'unique row count: {row_count}')
print(f'last saved offset: {last_offset}')
print(f'current filter: {current_filter}')
print(f'database path: {DEFAULT_DB_PATH}')


In [ ]:
export_all(db, DEFAULT_CSV_PATH, DEFAULT_PARQUET_PATH)


In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/oqmd_backups', exist_ok=True)
backup_db = '/content/drive/MyDrive/oqmd_backups/oqmd_half_heusler.db'
shutil.copy2(DEFAULT_DB_PATH, backup_db)
print(f'Backed up DB to {backup_db}')
